# grok-002 · 单卡 T4 + 真·LoRA（学习向）

> **你在学什么**：在 **1 张可见 T4** 上，对小 Instruct 模型做 **LoRA 指令微调** 的完整链路。
>
> **关键概念**：
> - `CUDA_VISIBLE_DEVICES=0`：Kaggle 套餐常是双 T4，用环境变量「只露出一张」
> - LoRA：冻结基座，只训低秩旁路（省显存、可分享 adapter）
> - chat template + causal LM loss：标准 SFT 形式
>
> **注意**：数据量很小，目的是 **学会流程**，不是刷榜。

## 建议对照实验
- 关掉 LoRA 只训最后一层 → 显存/效果差异  
- 对比微调前后同一探针问题的输出  


# grok-002-single-t4-qwen-lora

> **Slug/title:** `grok-002-single-t4-qwen-lora` — 1x T4 LoRA SFT on Qwen2.5-0.5B (CUDA_VISIBLE_DEVICES=0)


## 关于卡数（先把文档结论说清）

| Kaggle `machineShape` | 实际硬件 | 说明 |
|---|---|---|
| `NvidiaTeslaT4` | **Tesla T4 × 2** | 免费档常见的 T4 套餐（你日志里 `devices 2` 就是这个） |
| `NvidiaTeslaP100` | P100 × 1 | 单卡，但是 **P100 不是 T4** |
| 免费档 **没有** 单独的「T4 × 1」SKU | — | API/网页里选 T4 基本就是双卡包 |

**本 notebook 的策略（按你的要求：先把单卡 T4 做对）：**
1. 仍申请 `NvidiaTeslaT4`（否则拿不到 T4）
2. 在 **import torch 之前** 设置 `CUDA_VISIBLE_DEVICES=0`，进程里只暴露 **1 张 T4**
3. `assert torch.cuda.device_count() == 1` 且 `get_device_name(0)` 含 `T4`

双卡数据并行 / 更大模型留给后续 `grok-003`，不在这里硬上。

## 任务（有价值、且单卡 T4 足够）

对 **Qwen/Qwen2.5-0.5B-Instruct** 做 **LoRA 指令微调**（无 bitsandbytes 时用 fp16 LoRA，不走 4bit）：
- 自建小指令集（可复现）
- 可训练参数占比、峰值显存、loss 曲线
- **微调前/后** 同一探针生成对比
- 保存 adapter → `/kaggle/working/grok002_adapter`


In [ ]:
# 【步骤】锁定单卡：只让进程看见 GPU0（学习「逻辑单卡」）
# -*- cell 0: lock to ONE visible GPU BEFORE importing torch -*-
import os
# Kaggle free T4 SKU is dual-T4; pin to GPU0 so the job is effectively 1xT4.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
# quieter tokenizer forks
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

print('CUDA_VISIBLE_DEVICES=', os.environ.get('CUDA_VISIBLE_DEVICES'))


In [ ]:
# 【步骤】锁定单卡：只让进程看见 GPU0（学习「逻辑单卡」）
# -*- setup -*-
import os, json, time, math, random, platform, traceback, subprocess, sys
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

OUT = Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)

print('python', platform.python_version())
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available(), 'device_count', torch.cuda.device_count())
assert torch.cuda.is_available(), 'GPU required'
assert torch.cuda.device_count() == 1, f'expected 1 visible GPU after CUDA_VISIBLE_DEVICES=0, got {torch.cuda.device_count()}'
name = torch.cuda.get_device_name(0)
print('device[0]', name, 'mem_gb', round(torch.cuda.get_device_properties(0).total_memory/1e9, 2),
      'cap', torch.cuda.get_device_capability(0))
assert 'T4' in name, f'expected a Tesla T4, got {name}'

DEVICE = torch.device('cuda:0')  # 主设备：默认第一张可见 GPU
# T4 = SM 7.5 → no bf16 tensor cores; use fp16 AMP
AMP_DTYPE = torch.float16  # T4 无 fp16（无原生 bf16 tensor core）
print('amp_dtype', AMP_DTYPE)


In [ ]:
# 【步骤】学习率：warmup + cosine 退火
# -*- micro instruction set (no external dataset download) -*-
PAIRS = [
    ('What is 17 * 19?', '323'),
    ('If a train travels 120 km in 1.5 hours, average speed km/h?', '80'),
    ('Three primes between 20 and 40?', '23, 29, 31'),
    ('$80 item at 25% off. Sale price?', '60'),
    ('Minutes in 2.5 hours?', '150'),
    ('Next in 2,4,8,16,?', '32'),
    ('0.75 as percent?', '75%'),
    ('If x+7=20, x=?', '13'),
    ('GCD of 48 and 18?', '6'),
    ('Area of 8 by 5 rectangle?', '40'),
    ('Python function add(a,b) returning sum.', 'def add(a, b):\n    return a + b'),
    ('Python one-liner reverse list xs.', 'xs[::-1]'),
    ('JSON for Alice age 30.', '{"name": "Alice", "age": 30}'),
    ('Python: is s a palindrome?', 's == s[::-1]'),
    ('Max of three numbers a,b,c in Python.', 'def max3(a, b, c):\n    return max(a, b, c)'),
    ('SQL: all rows from users where active=1.', 'SELECT * FROM users WHERE active = 1;'),
    ('Explain gradient descent in one sentence.', 'Gradient descent updates parameters opposite the loss gradient to minimize the objective.'),
    ('Explain LoRA in one sentence.', 'LoRA freezes the base model and trains small low-rank adapters injected into linear layers.'),
    ('What is mixed-precision training?', 'Using fp16/bf16 compute while keeping master weights in fp32 for speed and lower memory.'),
    ('Why cosine LR decay?', 'It smoothly anneals the learning rate and often improves late-stage convergence.'),
    ('Translate to Chinese: Machine learning is powerful.', '机器学习很强大。'),
    ('What does PEFT stand for?', 'Parameter-Efficient Fine-Tuning'),
    ('Is 91 prime? If not, factor it.', 'No. 91 = 7 * 13.'),
    ('Sort ascending: 9, 2, 7, 2', '2, 2, 7, 9'),
    ('Capital of France?', 'Paris'),
    ('Binary of 13?', '1101'),
    ('2**10 = ?', '1024'),
    ('Average of 4, 8, 12?', '8'),
    ('Benefit of parameter-efficient fine-tuning?', 'Adapt large models with few trainable parameters and lower memory cost.'),
    ('(3+5)*2 = ? brief reason.', '16, because 3+5=8 and 8*2=16.'),
]
# light template expansion
DATA = []
for q,a in PAIRS:
    DATA.append((q, a))
    DATA.append((f'### Instruction:\n{q}\n### Response:', a))
random.Random(SEED).shuffle(DATA)
print('n_pairs', len(DATA))


In [ ]:
# 【步骤】检查有几张 GPU、名字是否为 Tesla T4
# -*- load Qwen2.5-0.5B-Instruct + LoRA (fp16 on 1xT4) -*-
# bitsandbytes often missing on Kaggle images; skip QLoRA and do standard LoRA in fp16.
# peft 0.19 may crash inside torchao dispatcher — we harden against that.

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
print('transformers', transformers.__version__)

import peft
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
print('peft', peft.__version__)

# Harden peft against broken torchao availability check (seen in logs).
try:
    import peft.tuners.lora.torchao as peft_torchao
    _orig = peft_torchao.is_torchao_available
    def _safe_torchao():
        try:
            return bool(_orig())
        except Exception:
            return False
    peft_torchao.is_torchao_available = _safe_torchao
    print('patched peft torchao availability check')
except Exception as e:
    print('torchao patch skipped:', e)

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
print('loading', MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# transformers v5 prefers `dtype=` over torch_dtype=
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map={'' : 0},
    trust_remote_code=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()  # 用算力换显存：激活重计算
if hasattr(model, 'enable_input_require_grads'):  # 梯度检查点时常需要，否则 LoRA 可能收不到梯度
    model.enable_input_require_grads()

# discover LoRA target modules present in this architecture
cand = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj','c_attn','W_pack']
present = sorted({n.split('.')[-1] for n,_ in model.named_modules() if n.split('.')[-1] in cand})
print('lora target modules:', present)
assert present, 'no LoRA targets found'

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=present,
)

try:
    model = get_peft_model(model, lora_cfg)
except Exception as e1:
    print('get_peft_model failed once:', repr(e1))
    # second chance: install torchao (optional) or retry after forcing flag false
    try:
        pip_install('torchao')
    except Exception as e_pip:
        print('torchao install failed:', e_pip)
    model = get_peft_model(model, lora_cfg)

model.print_trainable_parameters()
model.to(DEVICE)
print('model device', next(model.parameters()).device)
print('visible cuda devices', torch.cuda.device_count())


In [ ]:
# -*- dataset / collate -*-
def format_chat(instr, resp):
    # Qwen-style messages via apply_chat_template when available
    messages = [
        {'role': 'user', 'content': instr},
        {'role': 'assistant', 'content': resp},
    ]
    if hasattr(tokenizer, 'apply_chat_template'):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return f'<|user|>\n{instr}\n<|assistant|>\n{resp}{tokenizer.eos_token}'

class InstrDS(Dataset):
    def __init__(self, pairs, max_len=384):
        self.rows = []
        for instr, resp in pairs:
            text = format_chat(instr, resp)
            tok = tokenizer(text, truncation=True, max_length=max_len, padding=False)
            self.rows.append(tok)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        return self.rows[i]

def collate(batch):
    return tokenizer.pad(batch, padding=True, return_tensors='pt')

train_ds = InstrDS(DATA)
loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collate)
print('train rows', len(train_ds), 'batches/epoch', len(loader))


In [ ]:
# -*- generation helper + BEFORE probes -*-
PROBES = [
    'What is 17 * 19?',
    'Explain LoRA in one sentence.',
    'Write a Python function add(a, b) that returns the sum.',
]

@torch.no_grad()
def generate(prompt, max_new=80):
    model.eval()
    messages = [{'role': 'user', 'content': prompt}]
    if hasattr(tokenizer, 'apply_chat_template'):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = f'<|user|>\n{prompt}\n<|assistant|>\n'
    inputs = tokenizer(text, return_tensors='pt').to(DEVICE)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new,
        do_sample=False,  # greedy = more stable before/after compare  # greedy：评测可复现，避免采样噪声
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    gen = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

before = {p: generate(p) for p in PROBES}
print('=== BEFORE fine-tune ===')
for p,g in before.items():
    print('Q:', p)
    print('A:', g[:400])
    print('---')


In [ ]:
# 【步骤】混合精度 AMP：加速+省显存（T4 用 fp16）
# -*- train LoRA on 1xT4 -*-
trainable = [p for p in model.parameters() if p.requires_grad]
n_train = sum(p.numel() for p in trainable)
n_all = sum(p.numel() for p in model.parameters())
print(f'trainable {n_train:,} / {n_all:,} ({100*n_train/max(1,n_all):.3f}%)')

opt = torch.optim.AdamW(trainable, lr=2e-4, weight_decay=0.0)
steps = 100  # short but enough to move loss on 0.5B+LoRA
warmup = 10

def lr_at(step):
    if step < warmup:
        return (step + 1) / warmup
    t = (step - warmup) / max(1, steps - warmup)
    return 0.5 * (1.0 + math.cos(math.pi * t))

scaler = torch.amp.GradScaler('cuda', enabled=True)
torch.cuda.reset_peak_memory_stats()
model.train()

losses = []
t0 = time.perf_counter()
it = iter(loader)
for step in range(steps):
    try:
        batch = next(it)
    except StopIteration:
        it = iter(loader)
        batch = next(it)
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    labels = batch['input_ids'].clone()
    labels[batch['attention_mask'] == 0] = -100  # -100：CrossEntropy 忽略 pad（及可选 prompt）位置

    for g in opt.param_groups:
        g['lr'] = 2e-4 * lr_at(step)

    opt.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
        out = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], labels=labels)
        loss = out.loss
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(trainable, 1.0)  # 梯度裁剪：防爆炸
    scaler.step(opt)
    scaler.update()

    losses.append(float(loss.detach().float().cpu()))
    if step % 10 == 0 or step == steps - 1:
        print(f'step {step:03d}/{steps} loss={losses[-1]:.4f} lr={opt.param_groups[0]["lr"]:.2e}')

train_s = time.perf_counter() - t0
peak_gb = torch.cuda.max_memory_allocated() / 1e9
print(f'train_seconds={train_s:.1f} peak_mem_gb={peak_gb:.2f}')


In [ ]:
# 【步骤】锁定单卡：只让进程看见 GPU0（学习「逻辑单卡」）
# -*- AFTER probes + save artifacts -*-
after = {p: generate(p) for p in PROBES}
print('=== AFTER fine-tune ===')
for p,g in after.items():
    print('Q:', p)
    print('A:', g[:400])
    print('---')

adapter_dir = OUT / 'grok002_adapter'
adapter_dir.mkdir(exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print('saved adapter', adapter_dir)

results = {
    'notebook': 'grok-002',
    'phase': 'single_t4',
    'machine_shape_requested': 'NvidiaTeslaT4',  # Kaggle SKU (= dual T4 package)
    'cuda_visible_devices': os.environ.get('CUDA_VISIBLE_DEVICES'),
    'device_name': torch.cuda.get_device_name(0),
    'device_count_visible': torch.cuda.device_count(),
    'model_id': MODEL_ID,
    'mode': 'lora_fp16',  # not qlora: bitsandbytes absent
    'lora': {'r': 16, 'alpha': 32, 'targets': present},
    'steps': steps,
    'loss_start': losses[0],
    'loss_end': losses[-1],
    'loss_min': min(losses),
    'trainable_params': n_train,
    'total_params': n_all,
    'trainable_pct': 100.0 * n_train / max(1, n_all),
    'train_seconds': train_s,
    'peak_mem_gb': peak_gb,
    'before': before,
    'after': after,
    'artifact': str(adapter_dir),
    'notes': [
        'Kaggle free T4 accelerator is a 2xT4 SKU; this run forces 1 visible T4 via CUDA_VISIBLE_DEVICES=0.',
        'Dual-T4 DDP / larger models deferred to grok-003.',
        'bitsandbytes not used (module missing on image); fp16 LoRA is the correct single-T4 path for 0.5B.',
    ],
}
path = OUT / 'grok002_results.json'
path.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print('wrote', path)
print(json.dumps({k: results[k] for k in [
    'device_name','device_count_visible','mode','loss_start','loss_end',
    'trainable_pct','train_seconds','peak_mem_gb']}, indent=2))
print('DONE single-T4 LoRA')


## 学习检查清单

- 你应能回答：LoRA 训练哪些参数？为何要 chat template？单卡如何锁定？
- 建议：改一个超参重跑一小段，观察 log 变化（比只读代码更有效）。
